# 02 — Categories and Task Types

Ranked activity and completion summaries, retaining and extending the original spreadsheet reports.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run reporting-utils.ipynb

import sys
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))

In [ ]:
# Inclusive reporting period, based on task start date.
START_DATE = "2024-01-01"
END_DATE = "9999-12-31"
REPORT_DATE = date.today()

query = construct_query("task-history.sql", {"START-DATE": START_DATE, "END-DATE": END_DATE})
history = normalise_history(query_data(query))
period_label = f"{START_DATE} to {END_DATE} ({len(history):,} tasks)"
print(f"Reporting period: {period_label}")


## Category summary

In [ ]:
categories = grouped_summary(history, "Category")
categories

In [ ]:
figure, axes = plt.subplots(figsize=CHART_FIGURE_SIZE)
categories.sort_values("Task Count").plot.barh(
    x="Category", y="Task Count", legend=False, color=CHART_COLOR, ax=axes
)
format_chart(
    figure, axes, f"Tasks by category — {period_label}", "Tasks", "Category", grid_axis="x"
)
export_chart(get_export_folder_path(), "categories", "png", figure)
plt.show()

In [ ]:
figure, axes = plt.subplots(figsize=CHART_FIGURE_SIZE)
categories.sort_values("Completion Percentage").plot.barh(
    x="Category", y="Completion Percentage", legend=False, color=CHART_COLOR, ax=axes
)
axes.set_xlim(0, 100)
format_chart(
    figure, axes, f"Completion by category — {period_label}",
    "Completion percentage (%)", "Category", grid_axis="x"
)
export_chart(get_export_folder_path(), "category-completion", "png", figure)
plt.show()

## Task-type summary

In [ ]:
task_types = grouped_summary(history, ["Category", "Task Type"])
task_types

In [ ]:
TASK_TYPE_TOP_N = 15
ranked_task_types = task_types.head(TASK_TYPE_TOP_N).sort_values("Task Count")
figure, axes = plt.subplots(figsize=CHART_FIGURE_SIZE)
ranked_task_types.plot.barh(
    x="Task Type", y="Task Count", legend=False, color=CHART_COLOR, ax=axes
)
format_chart(
    figure, axes, f"Top {TASK_TYPE_TOP_N} task types — {period_label}",
    "Tasks", "Task type", grid_axis="x"
)
export_chart(get_export_folder_path(), "task-types", "png", figure)
plt.show()

In [ ]:
EXPORT_NAME = "02-categories-and-task-types.xlsx"
EXPORT_DATA = {"Categories": categories, "Task Types": task_types}
export_to_spreadsheet(
    get_export_folder_path(), EXPORT_NAME, EXPORT_DATA
)
print(f"Exported {EXPORT_NAME} to {get_export_folder_path()}")
